# Topic 8: Basics of Probability
**STOR 155 — Introduction to Data Models and Inference**

---

This notebook covers the core probability foundations from Topic 8. Every concept gets a **code simulation** so you can see the math in action.

**Key concepts:**
- Random phenomena & sample spaces
- Empirical vs. theoretical probability
- Law of Large Numbers
- Probability distributions
- Event operators (and, or, complement)
- Four basic rules of probability
- Benford's Law
- Independence & the multiplication rule

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

# Consistent style
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(155)  # STOR 155 :)

---
## 1. Random Phenomena & Terminology

A **random phenomenon** is a process whose outcome we cannot predict with certainty beforehand, but whose long-run behavior is predictable.

| Term | Definition |
|------|------------|
| **Random phenomenon** | A process with uncertain individual outcomes but predictable long-run patterns |
| **Sample space (Ω)** | The set of ALL possible outcomes |
| **Event** | A subset of the sample space (one or more outcomes) |
| **Probability** | A number between 0 and 1 assigned to each outcome/event |

**Examples of random phenomena:**
- Rolling a die
- Drawing a card
- Flipping a coin
- Tomorrow's temperature
- Whether a randomly chosen student passes STOR 155

---
## 2. Empirical vs. Theoretical Probability & Law of Large Numbers

There are two ways to think about probability:

- **Theoretical (classical):** Based on equally likely outcomes. $P(A) = \frac{\text{# outcomes in A}}{\text{# outcomes in } \Omega}$
- **Empirical (frequentist):** Based on running the experiment many times. $\hat{p}_n = \frac{\text{# times A occurred}}{n}$

The **Law of Large Numbers** says: as $n \to \infty$, $\hat{p}_n \to p$ (the empirical proportion approaches the true probability).

Let's see this with a die roll — what's the probability of rolling a 6?

In [ ]:
# Simulate rolling a die up to 100,000 times and track the running proportion of 6's
n_rolls = 100_000
rolls = np.random.randint(1, 7, size=n_rolls)
is_six = (rolls == 6)

# Running proportion: cumulative count of 6's / cumulative number of rolls
running_prop = np.cumsum(is_six) / np.arange(1, n_rolls + 1)

# Plot — this recreates the chart from the Topic 8 notes!
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(running_prop, color='steelblue', linewidth=0.8)
ax.axhline(y=1/6, color='red', linestyle='--', linewidth=1.5, label=f'True P(6) = 1/6 ≈ {1/6:.4f}')
ax.set_xscale('log')
ax.set_xlabel('n (number of rolls)', fontsize=12)
ax.set_ylabel('$\\hat{p}_n$ (proportion of 6\'s)', fontsize=12)
ax.set_title('Law of Large Numbers: Rolling a Die', fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim(0, 0.35)
plt.tight_layout()
plt.show()

print(f"After {n_rolls:,} rolls: p̂ = {running_prop[-1]:.4f}  (true p = {1/6:.4f})")

**Takeaway:** Early on, the proportion bounces around wildly. But as $n$ grows, it settles down to the true probability $p = 1/6$. This is exactly the Law of Large Numbers.

---
## 3. Building a Probability Distribution

Three steps:
1. Describe the **random phenomenon**
2. Describe the **sample space** (all possible outcomes)
3. Assign a **probability** to each outcome

**Key rule:** All probabilities must sum to 1.

### Example: Flip a fair coin 3 times

In [ ]:
# Sample space for 3 coin flips
sample_space = []
for c1 in ['H', 'T']:
    for c2 in ['H', 'T']:
        for c3 in ['H', 'T']:
            sample_space.append(c1 + c2 + c3)

print(f"Sample space Ω = {sample_space}")
print(f"|Ω| = {len(sample_space)} outcomes")
print(f"Each outcome has probability = 1/{len(sample_space)} = {1/len(sample_space):.4f}")

In [ ]:
# Probability distribution as a DataFrame
df_coins = pd.DataFrame({
    'Outcome': sample_space,
    'P(outcome)': [1/8] * 8
})

# Count number of heads in each outcome
df_coins['# Heads'] = df_coins['Outcome'].apply(lambda x: x.count('H'))
df_coins

In [ ]:
# Events: subsets of the sample space
event_at_least_2_heads = df_coins[df_coins['# Heads'] >= 2]
print("Event A = 'at least 2 heads':")
print(event_at_least_2_heads[['Outcome', '# Heads']].to_string(index=False))
print(f"\nP(A) = {len(event_at_least_2_heads)}/{len(df_coins)} = {len(event_at_least_2_heads)/len(df_coins):.4f}")

---
## 4. Venn Diagrams & Event Operators

Given events $A$ and $B$ within sample space $\Omega$:

| Operator | Notation | Meaning |
|----------|----------|--------|
| **And** (intersection) | $A \cap B$ or "$A$ and $B$" | Both A and B occur |
| **Or** (union) | $A \cup B$ or "$A$ or $B$" | A occurs, or B occurs, or both |
| **Complement** | $A^c$ | A does NOT occur |
| **Disjoint** | $A \cap B = \emptyset$ | A and B cannot both occur |

Let's visualize this with a card-drawing example.

In [ ]:
# Build a standard deck of 52 cards
suits = ['Clubs', 'Diamonds', 'Hearts', 'Spades']
ranks = ['Ace', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'Jack', 'Queen', 'King']
deck = pd.DataFrame([(s, r) for s in suits for r in ranks], columns=['Suit', 'Rank'])

# Define events
A = deck[deck['Suit'] == 'Diamonds']           # Event A: card is a diamond
B = deck[deck['Rank'].isin(['Jack', 'Queen', 'King'])]  # Event B: card is a face card

# Intersection: A AND B (diamond face cards)
A_and_B = deck[(deck['Suit'] == 'Diamonds') & (deck['Rank'].isin(['Jack', 'Queen', 'King']))]

# Union: A OR B
A_or_B = deck[(deck['Suit'] == 'Diamonds') | (deck['Rank'].isin(['Jack', 'Queen', 'King']))]

print(f"P(A) = P(Diamond)      = {len(A)}/52 = {len(A)/52:.4f}")
print(f"P(B) = P(Face card)    = {len(B)}/52 = {len(B)/52:.4f}")
print(f"P(A and B)             = {len(A_and_B)}/52 = {len(A_and_B)/52:.4f}")
print(f"P(A or B)              = {len(A_or_B)}/52 = {len(A_or_B)/52:.4f}")
print(f"\nCheck: P(A) + P(B) - P(A and B) = {len(A)/52 + len(B)/52 - len(A_and_B)/52:.4f}  ← matches P(A or B)!")

In [ ]:
# Visualize with a Venn-style bar chart
fig, ax = plt.subplots(figsize=(8, 4))

categories = ['Only A\n(Diamond, not face)', 'A and B\n(Diamond face card)', 'Only B\n(Face, not Diamond)', 'Neither']
only_A = len(A) - len(A_and_B)  # 10
both = len(A_and_B)              # 3
only_B = len(B) - len(A_and_B)  # 9
neither = 52 - only_A - both - only_B  # 30
counts = [only_A, both, only_B, neither]
colors = ['#5DADE2', '#2ECC71', '#F4D03F', '#BDC3C7']

bars = ax.bar(categories, counts, color=colors, edgecolor='black')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            f'{count}/52\n= {count/52:.3f}', ha='center', fontsize=10)

ax.set_ylabel('Number of cards', fontsize=12)
ax.set_title('Event Breakdown: Drawing from a Standard Deck', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Four Basic Rules of Probability

These are the foundation — everything else builds on them.

| # | Rule | Formula |
|---|------|---------|
| 1 | **Bounded** | $0 \leq P(A) \leq 1$ for any event $A$ |
| 2 | **Sample space** | $P(\Omega) = 1$ |
| 3 | **Addition (disjoint)** | If $A, B$ disjoint: $P(A \text{ or } B) = P(A) + P(B)$ |
| 4 | **Complement** | $P(A^c) = 1 - P(A)$ |

**General Addition Rule** (works for ANY two events, not just disjoint):

$$P(A \text{ or } B) = P(A) + P(B) - P(A \text{ and } B)$$

We subtract $P(A \text{ and } B)$ because it gets counted twice!

In [ ]:
# Demonstrate the complement rule with simulation
# Random phenomenon: roll a fair die
# Event A: roll a number > 4  →  P(A) = 2/6
# A^c: roll a number <= 4     →  P(A^c) = 4/6

n_sims = 100_000
rolls = np.random.randint(1, 7, size=n_sims)

p_A = np.mean(rolls > 4)
p_Ac = np.mean(rolls <= 4)

print(f"P(A)  = P(roll > 4)  = {p_A:.4f}  (theoretical: {2/6:.4f})")
print(f"P(A^c)= P(roll ≤ 4) = {p_Ac:.4f}  (theoretical: {4/6:.4f})")
print(f"P(A) + P(A^c)        = {p_A + p_Ac:.4f}  ← always equals 1!")

---
## 6. Benford's Law

In many real-world datasets, the **first digit** of numbers is NOT uniformly distributed!

Benford's Law predicts: $P(\text{first digit} = d) = \log_{10}\left(1 + \frac{1}{d}\right)$ for $d = 1, 2, ..., 9$

The course notes use **NC county populations** as an example. Let's recreate that.

In [ ]:
# NC County populations (2023 estimates, from the course notes)
nc_populations = [
    1190275, 1163701, 549866, 392821, 337690, 336892, 275901, 256452,
    241955, 240016, 238852, 237242, 213876, 199710, 179165, 175119,
    174804, 164645, 159964, 153661, 160626, 147458, 141477, 119230,
    116686, 117365, 106898, 102391, 101378, 96551, 95675, 92518,
    88338, 81624, 80574, 78970, 77001, 71482, 69615, 68521,
    67059, 66013, 65699, 65507, 62969, 62192, 59601, 54895,
    54748, 54446, 50121, 49520, 48832, 47298, 45532, 44893,
    44599, 44574, 44481, 42324, 42301, 41444, 39727, 38412,
    38110, 37774, 36473, 34376, 33549, 31593, 29959, 29484,
    27063, 26065, 22807, 22071, 21897, 21447, 20530, 20060,
    19453, 18938, 18836, 18636, 17561, 16715, 14998, 13916,
    13891, 13377, 12423, 11864, 11342, 11137, 10713, 10343,
    9401, 8052, 4607, 3461
]

# Extract first digits
first_digits = [int(str(pop)[0]) for pop in nc_populations]

# Observed proportions
observed = pd.Series(first_digits).value_counts().sort_index() / len(first_digits)

# Benford's Law predictions
digits = np.arange(1, 10)
benford = np.log10(1 + 1/digits)

# Compare
comparison = pd.DataFrame({
    'Digit': digits,
    'Observed': [observed.get(d, 0) for d in digits],
    "Benford's Law": benford
})
comparison['Observed'] = comparison['Observed'].round(3)
comparison["Benford's Law"] = comparison["Benford's Law"].round(3)
print(comparison.to_string(index=False))

In [ ]:
# Visualize the comparison
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(1, 10)
width = 0.35

bars1 = ax.bar(x - width/2, [observed.get(d, 0) for d in x], width, label='NC Counties (observed)', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, benford, width, label="Benford's Law (theoretical)", color='coral', edgecolor='black')

ax.set_xlabel('First Digit', fontsize=12)
ax.set_ylabel('Proportion', fontsize=12)
ax.set_title("Benford's Law: NC County Populations", fontsize=14)
ax.set_xticks(x)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Notice: digit 1 appears ~30% of the time, not 11% like you'd expect if uniform!")

In [ ]:
# Practice problems from the notes (using Benford's Law distribution)
# Let X = first digit of a randomly chosen number following Benford's Law

benford_dist = {d: np.log10(1 + 1/d) for d in range(1, 10)}

# P(X = 1 or X = 2)
p_1_or_2 = benford_dist[1] + benford_dist[2]  # Disjoint events!
print(f"P(X=1 or X=2) = {benford_dist[1]:.4f} + {benford_dist[2]:.4f} = {p_1_or_2:.4f}")

# P(X > 2) = 1 - P(X <= 2) = 1 - P(X=1 or X=2)  [complement rule]
p_gt_2 = 1 - p_1_or_2
print(f"P(X > 2) = 1 - {p_1_or_2:.4f} = {p_gt_2:.4f}")

# P(X is even) = P(X=2) + P(X=4) + P(X=6) + P(X=8)  [disjoint]
p_even = sum(benford_dist[d] for d in [2, 4, 6, 8])
print(f"P(X is even) = {p_even:.4f}")

# P(X=1 or X is even) = P(X=1) + P(X is even)  [disjoint since 1 is not even]
p_1_or_even = benford_dist[1] + p_even
print(f"P(X=1 or X is even) = {benford_dist[1]:.4f} + {p_even:.4f} = {p_1_or_even:.4f}")

---
## 7. Probability Distributions: Dice Example

**Random phenomenon:** Roll 3 fair dice and count the number of 6's. Let $X$ = count of 6's.

From the notes, the distribution is:

| $X$ | 0 | 1 | 2 | 3 |
|-----|---|---|---|---|
| $P(X)$ | .579 | .347 | .069 | .005 |

Let's verify this by simulation AND by exact calculation.

In [ ]:
# Exact calculation: each die has P(6) = 1/6, P(not 6) = 5/6
# X ~ Binomial(n=3, p=1/6)
from math import comb

p = 1/6
n = 3

print("Exact probabilities (Binomial formula):")
exact_probs = {}
for k in range(n + 1):
    prob = comb(n, k) * p**k * (1-p)**(n-k)
    exact_probs[k] = prob
    print(f"  P(X={k}) = C({n},{k}) × (1/6)^{k} × (5/6)^{n-k} = {prob:.4f}")

print(f"\nSum = {sum(exact_probs.values()):.4f}  ← must equal 1!")

In [ ]:
# Simulation verification
n_sims = 200_000
three_dice = np.random.randint(1, 7, size=(n_sims, 3))  # each row = one trial of 3 dice
count_sixes = np.sum(three_dice == 6, axis=1)  # count 6's in each trial

# Compare
print(f"{'X':>3} | {'Theoretical':>12} | {'Simulated':>12} | {'Notes':>8}")
print("-" * 45)
notes_probs = [0.579, 0.347, 0.069, 0.005]
for k in range(4):
    sim_prob = np.mean(count_sixes == k)
    print(f"{k:>3} | {exact_probs[k]:>12.4f} | {sim_prob:>12.4f} | {notes_probs[k]:>8.3f}")

In [ ]:
# Practice from notes: compute probabilities using this distribution
print("Practice Problems:")
print(f"  P(X=2 or X=3) = {exact_probs[2] + exact_probs[3]:.4f}  (disjoint events, just add)")
print(f"  P(X ≠ 0)      = 1 - P(X=0) = 1 - {exact_probs[0]:.4f} = {1 - exact_probs[0]:.4f}  (complement rule)")

---
## 8. SRS Example: Choosing 2 from 5 People

**Random phenomenon:** Take a simple random sample (SRS) of size 2 from {Alice, Bob, Cindy, Dave, Eva}.

This is a great example of a **non-uniform** starting point becoming **uniform** — each pair is equally likely.

In [ ]:
# Sample space: all combinations of 2 from 5
people = ['Alice', 'Bob', 'Cindy', 'Dave', 'Eva']
sample_space = list(combinations(people, 2))

print(f"Sample space ({len(sample_space)} outcomes):")
for pair in sample_space:
    print(f"  {set(pair)}")

print(f"\nEach pair has P = 1/{len(sample_space)} = {1/len(sample_space):.4f}")

In [ ]:
# Events from the notes:
# P(Bob is chosen but Cindy is not)
bob_no_cindy = [pair for pair in sample_space if 'Bob' in pair and 'Cindy' not in pair]
print(f"P(Bob chosen, Cindy not) = {len(bob_no_cindy)}/{len(sample_space)} = {len(bob_no_cindy)/len(sample_space):.4f}")
print(f"  Outcomes: {[set(p) for p in bob_no_cindy]}")

# P(Bob or Cindy is chosen)
bob_or_cindy = [pair for pair in sample_space if 'Bob' in pair or 'Cindy' in pair]
print(f"\nP(Bob or Cindy chosen) = {len(bob_or_cindy)}/{len(sample_space)} = {len(bob_or_cindy)/len(sample_space):.4f}")
print(f"  Outcomes: {[set(p) for p in bob_or_cindy]}")

---
## 9. Independence & the Multiplication Rule

Two events $A$ and $B$ are **independent** if knowing that $A$ occurred does NOT change the probability of $B$.

**Multiplication rule for independent events:**
$$P(A \text{ and } B) = P(A) \times P(B)$$

**Key distinction:**
- **Disjoint** = A and B CANNOT happen together → $P(A \text{ and } B) = 0$
- **Independent** = A and B DON'T AFFECT each other → $P(A \text{ and } B) = P(A) \cdot P(B)$

Disjoint events with non-zero probabilities are actually DEPENDENT (if A happens, B can't)!

In [ ]:
# Example: Roll a fair die TWICE
# Event A: first roll = 6
# Event B: second roll = 6
# Are A and B independent?  YES — the dice don't affect each other

n_sims = 200_000
roll1 = np.random.randint(1, 7, size=n_sims)
roll2 = np.random.randint(1, 7, size=n_sims)

p_A = np.mean(roll1 == 6)
p_B = np.mean(roll2 == 6)
p_A_and_B = np.mean((roll1 == 6) & (roll2 == 6))

print(f"P(A) = P(first=6)           = {p_A:.4f}   (theoretical: {1/6:.4f})")
print(f"P(B) = P(second=6)          = {p_B:.4f}   (theoretical: {1/6:.4f})")
print(f"P(A and B)                  = {p_A_and_B:.4f}   (theoretical: {1/36:.4f})")
print(f"P(A) × P(B)                = {p_A * p_B:.4f}   (theoretical: {1/36:.4f})")
print(f"\nP(A and B) ≈ P(A)×P(B)?  → {'YES — independent!' if abs(p_A_and_B - p_A*p_B) < 0.005 else 'No'}")
print(f"\nAre A and B disjoint?     → NO! Both dice CAN show 6.")

In [ ]:
# Contrast: SRS example — are events independent?
# Event A: Alice is selected.  Event B: Bob is selected.
# In our SRS of 2 from 5 people:

p_alice = sum(1 for pair in sample_space if 'Alice' in pair) / len(sample_space)
p_bob = sum(1 for pair in sample_space if 'Bob' in pair) / len(sample_space)
p_alice_and_bob = sum(1 for pair in sample_space if 'Alice' in pair and 'Bob' in pair) / len(sample_space)

print(f"P(Alice) = {p_alice:.4f}")
print(f"P(Bob)   = {p_bob:.4f}")
print(f"P(Alice and Bob)     = {p_alice_and_bob:.4f}")
print(f"P(Alice) × P(Bob)   = {p_alice * p_bob:.4f}")
print(f"\nP(A∩B) ≠ P(A)×P(B)  → NOT independent!")
print(f"\nIntuition: If Alice is picked, that uses one of the 2 slots,")
print(f"making it MORE likely Bob fills the remaining slot (1/4 vs 2/5).")

---
## 10. General Addition Rule — Putting It All Together

The **General Addition Rule** works for ANY two events:

$$P(A \text{ or } B) = P(A) + P(B) - P(A \text{ and } B)$$

When events are disjoint, $P(A \text{ and } B) = 0$, so it simplifies to Rule 3.

In [ ]:
# General Addition Rule with dice
# Roll two dice. A = sum is 7, B = first die is 4

# Build the sample space
omega = [(d1, d2) for d1 in range(1,7) for d2 in range(1,7)]

A = [(d1,d2) for d1,d2 in omega if d1+d2 == 7]
B = [(d1,d2) for d1,d2 in omega if d1 == 4]
A_and_B = [(d1,d2) for d1,d2 in omega if d1+d2 == 7 and d1 == 4]
A_or_B = [(d1,d2) for d1,d2 in omega if d1+d2 == 7 or d1 == 4]

p_A = len(A)/36
p_B = len(B)/36
p_AandB = len(A_and_B)/36
p_AorB = len(A_or_B)/36

print(f"P(A) = P(sum=7)       = {len(A)}/36 = {p_A:.4f}")
print(f"P(B) = P(first=4)     = {len(B)}/36 = {p_B:.4f}")
print(f"P(A and B) = P(4,3)   = {len(A_and_B)}/36 = {p_AandB:.4f}")
print(f"")
print(f"General Addition Rule:")
print(f"P(A or B) = P(A) + P(B) - P(A and B)")
print(f"         = {p_A:.4f} + {p_B:.4f} - {p_AandB:.4f}")
print(f"         = {p_A + p_B - p_AandB:.4f}")
print(f"")
print(f"Direct count: {len(A_or_B)}/36 = {p_AorB:.4f}  ✓")

---
## Quick Reference: Topic 8 Formula Sheet

| Concept | Formula |
|---------|---------|
| Theoretical probability | $P(A) = \frac{|A|}{|\Omega|}$ (when outcomes equally likely) |
| Empirical probability | $\hat{p}_n = \frac{\text{count of A}}{n}$ |
| Law of Large Numbers | $\hat{p}_n \to p$ as $n \to \infty$ |
| Complement | $P(A^c) = 1 - P(A)$ |
| Addition (disjoint) | $P(A \cup B) = P(A) + P(B)$ |
| General Addition | $P(A \cup B) = P(A) + P(B) - P(A \cap B)$ |
| Multiplication (independent) | $P(A \cap B) = P(A) \cdot P(B)$ |
| Benford's Law | $P(d) = \log_{10}(1 + 1/d)$ |

**Key distinction:** Disjoint ≠ Independent
- Disjoint: can't both happen → $P(A \cap B) = 0$
- Independent: don't affect each other → $P(A \cap B) = P(A) \cdot P(B)$